# Dust Mapping

This notebook acts as a playground to study clusters effected by extinction, motivated by the study of Cl Tombaugh 4.

After selecting this cluster and noticing the severe color shift and dimming caused by the presence of dust, the focus was shifted towards creating a base with the red clump to study dust column density and extinction as a function of distance, resulting in 3D maps of dust and extinction.

This project originally featured a section where we analyzed our entire dataset and grouped into galactic components for the sake of a visualization of where different groups sat on a CMD. However, this went star-by-star in a very disorganized fashion (as it was an earlier project in my coding journey), which is inefficient. So we utilized object-oriented programming to solve this.

Note: Due to the nature of this course and my skill level at the time, I was advised to use generative AI to assist with coding for this project, which was used primarily with plotting

# Selections:

In [ ]:
import numpy as np
import matplotlib as plt
from matplotlib import pyplot
import matplotlib.pyplot as plt
from astropy.table import Table as tb
from astropy.io import votable
import seaborn as sns
import pandas as pd
from matplotlib.patches import Ellipse
import astropy.coordinates as coord
import astropy.units as u
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
from scipy.integrate import simpson
from scipy.stats import gaussian_kde
import plotly.graph_objects as go
from scipy.stats import binned_statistic_2d
import plotly.express as px

from cluster_selection import ClusterSelector, absolute_magnitude # my functions!

In [ ]:
# importing my data
tom_data = tb.read('Data/YEAH-result.vot', format = 'votable') # Obtained from Gaia Archive with ADQL

full_data = tb.read('Data/notebook.fits', format = 'fits') # Provided by advisor for this project

In [ ]:
# Isolating the Cl Tombaugh 4 cluster from the full dataset using cluster_selector.py to create new objects

tom_selector1 = ClusterSelector(tom_data)
tom_round1 = tom_selector1.select() # empty call just gives us the entire dataset

tom_selector2 = ClusterSelector(tom_round1)
tom_round2 = tom_selector2.select(ra_range=(37.2, 37.4), dec_range=(61.7, 61.9), par_mask=True)

tom_selector3 = ClusterSelector(tom_round2)
tom_round3 = tom_selector3.select(ra_range=(37.2, 37.4), dec_range=(61.7, 61.9), sigma_window=['pm',0.3], ref_round=tom_round2, par_mask=True)

tom_labels, tom_dist, tom_galcen, _ = tom_round3.galactic_comp() # ignoring R value

In [ ]:
full_selector = ClusterSelector(full_data)
full_round1 = full_selector.select(radvel_mask=True, par_mask=True) # limit full dataset to only measured radial velocities and well-measured parallax

full_labels, full_dist, full_galcen, _ = full_round1.galactic_comp()

## Looking at Cl Tombaugh 4

In [ ]:
plt.scatter(tom_round1.color, absolute_magnitude(tom_round1.parallax, tom_round1.mag), c='royalblue', s=2, label='Round 1')
plt.scatter(tom_round2.color, absolute_magnitude(tom_round2.parallax, tom_round2.mag), c='black', s=2, label='Round 2')
plt.scatter(tom_round3.color, absolute_magnitude(tom_round3.parallax, tom_round3.mag), c='deeppink', s=2, label='Round 3')

#colorbar1 = plt.colorbar(tom_scatter, label = 'BP/RP Color Index')
plt.title('Cl Tombaugh 4 - Color-Magnitude Diagram')
plt.xlabel('Color')
plt.ylabel('Absolute Magnitude')

plt.gca().invert_yaxis()
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Analyzing Galaxy’s Components + Looking for Red Clump

In [ ]:
full_dm = 5*np.log10(full_dist.pc/10.0) # distance modulus

red_one = (full_round1.color < 3) & (full_round1.color >= 1) & (full_round1.mag - full_dm < 3) & (full_round1.mag - full_dm >= 0) & (abs(full_galcen.z.value) < 200.0) & (full_dist.pc < 1000) & (full_dist.pc > 500)

red_selector = ClusterSelector(full_round1)
red_round1 = red_selector.select(add_mask=red_one)

red_labels, red_dist, red_galcen, red_R = red_round1.galactic_comp()

In [ ]:
red_dm = 5*np.log10(red_dist.pc/10.0)

# hardcoded to look at the red branch
x = red_round1.color
y = 1.5*x -1.1

std_dev = 0.75 * np.std(y)
max_y = y + std_dev
min_y = y - std_dev

branch = (red_round1.mag-red_dm <= max_y) & (red_round1.mag-red_dm >= min_y)

branch_selector = ClusterSelector(red_round1)
branch_round1 = branch_selector.select(add_mask=branch)

branch_labels, branch_dist, branch_galcen, branch_R = branch_round1.galactic_comp()

# print(np.isnan(std_dev))
# print(len(branch_round1))

In [ ]:
branch_dm = 5*np.log10(branch_dist.pc/10.0)
tom_abs_mag = absolute_magnitude(tom_round2.parallax, tom_round2.mag)

color_plain = np.ma.filled(np.ma.masked_invalid(full_round1.color), np.nan) # catching nans/infs without altering shape
full_abs_mag = full_round1.mag - full_dm

fig, ax = plt.subplots()

component_styles = {"thin_disk": ("blue", "Thin Disk"), "thick_disk": ("cyan", "Thick Disk"), "halo": ("lawngreen", "Halo"),}

for label, (color, display_name) in component_styles.items():
    mask = (full_labels == label) & np.isfinite(color_plain) & np.isfinite(full_abs_mag)
    ax.plot(color_plain[mask], abs_mag[mask], ',', c=color, alpha=0.1, label=display_name)

ax.scatter(tom_round2.color, tom_abs_mag, s=0.1, alpha=0.7, c='magenta', label='Tombaugh 4 ')

ax.set_xlabel('BP-RP')
ax.set_ylabel('Absolute Magnitude $M_G$')
ax.set_title('Observed CMD for Thin/Thick Disk and Halo Components')
ax.invert_yaxis()

legend_handles = [Line2D([0], [0], marker='o', color='white', markerfacecolor=color, markersize=5, alpha=0.8, label=name) for label, (color, name) in component_styles.items()]
legend_handles.append(Line2D([0], [0], marker='o', color='white', markerfacecolor='magenta', markersize=5, alpha=0.8, label='Tombaugh 4'))
ax.legend(handles=legend_handles)
plt.show()

In [ ]:
# Background CMD as a 2D histogram to highlight our selected red clump!

finite_mask = np.isfinite(color_plain) & np.isfinite(full_abs_mag)

fig, ax = plt.subplots()
counts, xedges, yedges, im = ax.hist2d(color_plain[finite_mask], abs_mag_background[finite_mask], bins=100, cmap='plasma', norm=LogNorm())

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Counts per Bin')

ax.set_xlabel('BP-RP')
ax.set_ylabel('$M_G$')
ax.set_title('Color-Magnitude Diagram as 2D Histogram')

# red clump reference point (PARSEC models, Ruiz-Dern et al. 2018)
ax.add_patch(plt.Circle((1.23, 0.51), radius=0.25, color='black', fill=False, alpha=1, zorder=6))
ax.plot(1.23, 0.51, 'k.', zorder=7)

branch_abs_mag = branch_round1.mag - branch_dm
ax.scatter(branch_round1.color, branch_abs_mag, c='mediumturquoise', alpha=0.3, s=1,
           label='Reddened Selection', zorder=5)

ax.invert_yaxis()

ax.legend(loc='lower right')
plt.show()

In [ ]:
# sanity check for the reddened red clump selection, performance evaluation

bin_edges = np.arange(min(branch_round1.mag), max(branch_round1.mag), 0.1) 

star_counts = np.histogram(branch_round1.mag, bins=bin_edges)

plt.figure(figsize=(8, 6))
plt.hist(branch_round1.mag, bins=bin_edges, histtype='stepfilled', color='royalblue')
plt.xlabel('Apparent Magnitude (G)')
plt.ylabel('Number of Stars')
plt.title('Luminosity Function (Apparent Magnitude)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
disk3 = np.where((abs(full_galcen.z.value)<800.0)&(full_dist.pc<1000))

plt.scatter(full_round1.ra[disk3],full_round1.dec[disk3], c='black', alpha=0.01,s=0.02)
plt.scatter(branch_round1.ra, branch_round1.dec, c='royalblue', alpha=0.1, s=0.1, label='Reddened Selection')
plt.ylabel('Declination (degrees)')
plt.xlabel('RA (degrees)')
plt.title('RA vs. Declination of Red Clump')
plt.show()

In [ ]:
thin_disk = full_labels == "thin_disk"
thick_disk = full_labels == "thick_disk"
halo = full_labels == "halo"

tom = tom_labels == "thin_disk"

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(full_round1.ra[thin_disk], full_round1.dec[thin_disk], s=1, alpha=0.01, color='royalblue', label="Thin Disk")

ax.scatter(full_round1.ra[thick_disk], full_round1.dec[thick_disk], s=1, alpha=0.01, color='black', label="Thick Disk")

ax.scatter(full_round1.ra[halo], full_round1.dec[halo], s=1, alpha=0.01, color='lawngreen', label="Halo")

#print(tom_labels)

ax.set_xlabel("RA (degrees)")
ax.set_ylabel("Dec (degrees)")
ax.legend()
plt.show()

In [ ]:
# 2D density map based on distribution of stars in galactic plane.
# Doing this because we identified the red clump as standard candles, so we can map how that stellar density is distributed
density, x_edges, y_edges, _ = binned_statistic_2d(branch_R, np.abs(branch_galcen.z.value), values=None, statistic='count', bins=(50, 50))

plt.figure()
plt.imshow(np.log10(density.T), origin='lower', extent=(x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]), aspect='auto', cmap='plasma')
plt.colorbar(label='log10(Stellar Count)')
plt.xlabel('R (kpc)')
plt.ylabel('|z| (kpc)')
plt.title('Galactic Density Map')

plt.show()

## Extinction and Dust Mapping

In [ ]:
# Apply Extinction and Reddening to tombaugh 4 ---  first estimates

R_v = 3.1 # ratio for Milky Way, determine with extinction laws
k = 1.85 # standard coefficient for Milky Way
g = 2.27
    
color_excess = (branch_round1.color - 1.23)/k # in B-V
reddening = branch_round1.color - 1.23 # in BP-RP

A_v = color_excess * R_v
A_g = color_excess * g

In [ ]:
plt.scatter(1000/branch_round1.parallax, A_g, c='darkblue', s=0.5, alpha=0.15)
plt.xlabel('Distance (pc)')
plt.ylabel('Extinction') 
plt.title('Extinction as a Function of Distance')
plt.show()

In [ ]:
branch_dm = 5 * np.log10(branch_dist.pc / 10.0)
#print(red_dist)

# estimated that red clump stars sit in a box with a central abs mag of M = 0.51 ± 0.25 mag
# and a central colour of GBP − GRP = 1.23 ± 0.05 mag. 

kde = gaussian_kde([branch_round1.color, branch_abs_mag])
density = kde([branch_round1.color, branch_abs_mag])

scatter = plt.scatter(branch_round1.color, branch_abs_mag, alpha=0.05, s=0.1, c=density, cmap='plasma')

x=[1.23]
y=[0.51]
x_err=[0.05]
y_err=[0.25]

for xi, yi, xe, ye in zip(x, y, x_err, y_err):
    circle = plt.Circle((xi, yi), radius=max(xe, ye), color='red', fill=False, alpha=0.5)
    plt.gca().add_patch(circle)


plt.plot(1.23,0.51, 'ro')
plt.gca().invert_yaxis()

plt.xlabel('BP-RP')
plt.ylabel('Absolute Mag $M_G$')
plt.title('CMD For Red Clump')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
d = 1/branch_round1.parallax
rads = np.pi/180
x = d * np.cos(branch_round1.dec * rads) * np.cos(branch_round1.ra * rads)
y = d * np.cos(branch_round1.dec * rads) * np.sin(branch_round1.ra * rads)
z = d * np.sin(branch_round1.dec * rads)

alpha_values = (A_g - A_g.min()) / (A_g.max() - A_g.min())

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(x, y, z, c=A_g, cmap='viridis', s=1, alpha=alpha_values)
fig.colorbar(sc, label="Extinction (A_G)")

ax.set_xlabel('X (kpc)')
ax.set_ylabel('Y (kpc)')
ax.set_zlabel('Z (kpc)')
ax.set_title('3D Dust Extinction Map')
plt.show()

In [ ]:
d = 1/branch_round1.parallax
rads = np.pi/180
x = d * np.cos(branch_round1.dec * rads) * np.cos(branch_round1.ra * rads)
y = d * np.cos(branch_round1.dec * rads) * np.sin(branch_round1.ra * rads)
z = d * np.sin(branch_round1.dec * rads)

alpha_values = (A_g - A_g.min()) / (A_g.max() - A_g.min())
fig = go.Figure(data=go.Scatter3d(x=x, y=y, z=z, mode='markers', marker = dict(size=0.85, color=A_g, colorscale='Viridis', colorbar = dict(title="A_g", tickvals=np.linspace(A_g.min(), A_g.max(), 5), tickformat=".2f"),), opacity=1))

fig.update_layout(scene=dict(xaxis_title="X (kpc)", yaxis_title="Y (kpc)", zaxis_title="Z (kpc)"), title="3D Dust Extinction Map")
fig.show()

In [ ]:
data = pd.DataFrame({"RA": branch_round1.ra, "Dec": branch_round1.dec, "Distance (kpc)": 1/branch_round1.parallax, "Extinction (A_V)": A_g})

fig = px.scatter_3d(data, x="RA", y="Dec", z="Distance (kpc)", color="Extinction (A_V)", color_continuous_scale="inferno", title="Interactive 3D Extinction Map")

fig.update_traces(marker=dict(size=1))
fig.update_layout(scene=dict(xaxis_title="RA (deg)", yaxis_title="Dec (deg)", zaxis_title="Distance (kpc)"))

fig.show()